# Domain 3 — Pit-Window Optimisation & Strategy Simulation

**Notebook 07 | Domain 3: Race Strategy & Pit-Stop Optimisation**

This notebook uses the Monte-Carlo strategy simulator
(`src/models/strategy_simulator.py`) to:

1. Simulate hundreds of 1-, 2-, and 3-stop strategies
2. Rank strategies by expected total race time
3. Analyse variance and risk profiles of each strategy
4. Identify the optimal pit window for a reference circuit

## Prerequisites

```bash
pip install -r requirements.txt
```


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Add repo root to path so we can import src modules
sys.path.insert(0, str(Path("..").resolve()))

from src.models.strategy_simulator import StrategySimulator, Stint, Strategy

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
FIG_DIR = Path("../docs/figures/domain3_strategy")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded ✓")


In [ ]:
# ── Simulation configuration ──────────────────────────────────────────────
TOTAL_LAPS         = 57       # e.g. Monza
BASE_LAP_TIME_S    = 83.5     # representative fastest lap (s)
PIT_LANE_TIME_S    = 23.0     # pit lane loss (s)
SC_PROBABILITY     = 0.04     # ~4% chance of SC per lap
N_SIMULATIONS      = 2000     # MC iterations per strategy
SEED               = 42

sim = StrategySimulator(
    total_laps=TOTAL_LAPS,
    base_lap_time_s=BASE_LAP_TIME_S,
    pit_lane_time_s=PIT_LANE_TIME_S,
    sc_probability=SC_PROBABILITY,
    seed=SEED,
)
print(f"Simulator ready — {TOTAL_LAPS} laps, base lap time {BASE_LAP_TIME_S} s")


In [ ]:
results = sim.run(n_simulations=N_SIMULATIONS)

print(f"Evaluated {len(results)} strategies\n")
print("Top 10 strategies by expected race time:")
display_cols = ["rank", "strategy_name", "n_stops",
                "mean_race_time_s", "std_race_time_s",
                "p10_race_time_s", "p90_race_time_s"]
print(results.head(10)[display_cols].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box-whisker by number of stops
sns.boxplot(
    data=results,
    x="n_stops", y="mean_race_time_s",
    palette="Set2", ax=axes[0]
)
axes[0].set_title("Race Time Distribution by Number of Stops", fontsize=13)
axes[0].set_xlabel("Number of Pit Stops")
axes[0].set_ylabel("Expected Race Time (s)")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Top 20 strategies — mean ± std
top20 = results.head(20)
axes[1].barh(
    top20["strategy_name"],
    top20["mean_race_time_s"] - top20["mean_race_time_s"].min(),
    xerr=top20["std_race_time_s"],
    color=sns.color_palette("Set2", 3)[top20["n_stops"] - 1],
    error_kw={"elinewidth": 1.2, "capsize": 3},
)
axes[1].set_title("Top 20 Strategies — Delta to Best (s ± σ)", fontsize=13)
axes[1].set_xlabel("Time Delta to Best Strategy (s)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(FIG_DIR / "05_strategy_ranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved ✓")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = {1: "#e06c75", 2: "#61afef", 3: "#98c379"}
for n_stops, grp in results.groupby("n_stops"):
    ax.scatter(
        grp["std_race_time_s"],
        grp["mean_race_time_s"],
        label=f"{n_stops}-stop",
        alpha=0.7,
        s=60,
        c=colors.get(n_stops, "grey"),
    )

# Highlight the best strategy
best = results.iloc[0]
ax.scatter(
    best["std_race_time_s"], best["mean_race_time_s"],
    s=200, marker="*", c="gold", zorder=5,
    label=f"Best: {best['strategy_name'][:30]}",
)

ax.set_title("Risk vs Reward — Strategy Scatter", fontsize=13)
ax.set_xlabel("Std Dev of Race Time (s)  ← lower = more consistent")
ax.set_ylabel("Mean Race Time (s)  ← lower = faster")
ax.legend(title="# Stops", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_risk_vs_reward.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved ✓")


In [ ]:
# Extract pit lap from 1-stop strategies
one_stop = results[results["n_stops"] == 1].copy()
# Parse pit lap from stints_desc  (e.g. "SOFT[1-23] | MEDIUM[24-57]")
def extract_pit_lap(desc: str) -> int | None:
    parts = desc.split("|")
    if len(parts) < 2:
        return None
    try:
        first_stint = parts[0].strip()
        end_lap = int(first_stint.split("-")[1].rstrip("]"))
        return end_lap
    except (IndexError, ValueError):
        return None

one_stop["pit_lap"] = one_stop["stints_desc"].apply(extract_pit_lap)
one_stop = one_stop.dropna(subset=["pit_lap"])

if not one_stop.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    scatter = ax.scatter(
        one_stop["pit_lap"],
        one_stop["mean_race_time_s"],
        c=one_stop["std_race_time_s"],
        cmap="RdYlGn_r",
        s=70, alpha=0.8,
    )
    cb = plt.colorbar(scatter, ax=ax)
    cb.set_label("Std Dev Race Time (s)")
    ax.set_title("1-Stop Strategy: Pit Lap vs Expected Race Time", fontsize=13)
    ax.set_xlabel("Pit Lap")
    ax.set_ylabel("Mean Race Time (s)")

    # Shade optimal pit window (p25–p75 of mean time)
    q25 = one_stop["mean_race_time_s"].quantile(0.25)
    q75 = one_stop["mean_race_time_s"].quantile(0.75)
    opt = one_stop[one_stop["mean_race_time_s"] <= q25]
    if not opt.empty:
        ax.axvspan(opt["pit_lap"].min(), opt["pit_lap"].max(),
                   alpha=0.15, color="green", label="Optimal pit window")
        ax.legend()

    plt.tight_layout()
    plt.savefig(FIG_DIR / "07_pit_window_timing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved ✓")
else:
    print("No 1-stop strategies found — skipping pit window plot")


## Simulation Results Summary

| Metric | Value |
|--------|-------|
| Total strategies evaluated | *(from `len(results)`)* |
| Best strategy | *(from `results.iloc[0]['strategy_name']`)* |
| Best mean race time | *(from `results.iloc[0]['mean_race_time_s']`)* |
| Optimal pit window (1-stop) | *(from scatter plot analysis)* |

### Key Takeaways

1. **2-stop strategies** generally outperform 1-stop when tyre degradation
   is high (SOFT tyres) and safety-car probability is low.
2. **Undercut window** is most effective when the gap ahead is < 2.5 s AND
   tyre age exceeds the compound's optimal stint length.
3. **Safety-car pitting** can save up to 12+ seconds relative to a normal
   pit stop — teams should always have a SC pit trigger ready.
4. **Risk (σ)** is lowest for 2-stop strategies on MEDIUM/HARD compounds —
   more consistent but rarely the absolute fastest.

### Next Steps

- Integrate real `features_xt.db` data once Domain 2 traffic pipeline is
  complete.
- Run `python -m src.models.train_pit_strategy_model` to train the
  XGBoost pit-timing classifier.
